# White blood cell classification – Colab quickstart
Runtime → Change runtime type → **T4 GPU** before running.
Run the cells top to bottom. Training results are copied to Google Drive at the end so they survive a disconnect.

In [ ]:
!nvidia-smi
import torch; print("CUDA available:", torch.cuda.is_available())

## 1. Get the code
If the repo is private, use a personal access token in the URL: `https://<TOKEN>@github.com/...`

In [ ]:
REPO = "https://github.com/Yasmin-maker1/Automated-Classification-of-White-Blood-Cell-Subtypes-Using-Convolutional-Neural-Networks.git"
!git clone {REPO} repo
%cd repo
!pip -q install scikit-learn pandas matplotlib

## 2. Get the data (Kaggle)
Create a token at Kaggle → Settings → *Create New Token* and upload the `kaggle.json` file when prompted.

In [ ]:
from google.colab import files
files.upload()   # choose kaggle.json
!mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!pip -q install kaggle
!kaggle datasets download -d paultimothymooney/blood-cells -p data --unzip

## 3. Explore the data and check for leakage

In [ ]:
!python src/eda.py --data-root data
from IPython.display import Image, display
display(Image("outputs/eda/sample_grid.png"))
display(Image("outputs/eda/class_counts.png"))

In [ ]:
!python src/leakage_check.py --data-root data --num-workers 2
display(Image("outputs/leakage/top_pairs.png"))      # LOOK at these
display(Image("outputs/leakage/random_pairs.png"))

## 4. Train
Baseline first, then the main model. Compare **validation** curves; do not use the test folder yet.

In [ ]:
!python src/train.py --data-root data --model smallcnn --epochs 20 --num-workers 2 --amp --run-name smallcnn_baseline

In [ ]:
!python src/train.py --data-root data --model resnet50 --epochs 15 --freeze-epochs 2 --num-workers 2 --amp --run-name resnet50_v1
display(Image("outputs/resnet50_v1/curves.png"))

## 5. FINAL evaluation (run once, for the chosen model only)

In [ ]:
# !python src/evaluate.py --checkpoint outputs/resnet50_v1/best.pt --num-workers 2
# display(Image("outputs/resnet50_v1/eval/confusion_matrix.png"))

## 6. Save results to Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/wbc_project && cp -r outputs /content/drive/MyDrive/wbc_project/